# S5 J3 — Redis pour systèmes IA en production

Notebook étudiant généré à partir des sources Markdown du jour. Les corrigés sont exclus de ce notebook.

# Objectifs pédagogiques

## Objectifs principaux

À la fin de ce jour, l'apprenant doit être capable de :

1. expliquer le rôle de Redis dans une architecture IA de production ;
2. distinguer cache, état court terme, verrou, idempotence et rate limiting ;
3. concevoir une stratégie de clés Redis lisible et maintenable ;
4. implémenter un cache LLM en stratégie cache-aside ;
5. stocker un état conversationnel court avec expiration ;
6. construire un rate limiter simple et atomique ;
7. intégrer Redis dans une API FastAPI ;
8. identifier les limites de Redis pour les données durables ;
9. appliquer les bonnes pratiques de sécurité autour des prompts, réponses et données utilisateur ;
10. produire un lab exécutable et testable.

## Compétences AI Engineering ciblées

- Architecture de systèmes IA à latence contrôlée.
- Réduction des coûts d'appels LLM.
- Gestion d'état temporaire pour agents et assistants.
- Protection d'une API contre les abus et les pics de charge.
- Séparation entre stockage volatil et stockage durable.
- Design de clés et TTL en environnement multi-utilisateur.

## Pré-requis

L'apprenant doit avoir compris :

- les principes d'une API FastAPI ;
- les appels LLM simulés ou réels ;
- les notions de requête, réponse, latence et coût ;
- les bases de Python asynchrone ;
- les risques liés à la persistance de données sensibles.

## Critères de réussite

La journée est réussie si l'apprenant peut :

- expliquer pourquoi un cache LLM peut être dangereux s'il ignore l'utilisateur ou le contexte ;
- coder une fonction `get_or_set` avec TTL ;
- justifier un TTL différent pour une réponse LLM, une session et un compteur de rate limit ;
- définir une convention de clés Redis ;
- exécuter le lab localement avec Docker Compose.


## Extrait du chapitre

# Chapitre — Redis dans une plateforme IA de production

## 1. Pourquoi Redis apparaît en production IA

Un prototype IA peut appeler directement un modèle à chaque requête. En production, ce modèle naïf devient rapidement coûteux, lent et fragile.

Les appels LLM sont généralement :

- plus chers qu'une lecture mémoire ;
- plus lents qu'une lecture locale ;
- dépendants d'un fournisseur externe ;
- sensibles aux pics de trafic ;
- parfois redondants lorsque plusieurs utilisateurs demandent la même transformation.

Redis répond à ces problèmes par une couche mémoire rapide, simple et opérationnelle. Dans cette journée, on l'utilise pour quatre besoins :

1. cache de réponses ;
2. état conversationnel court ;
3. limitation de débit ;
4. idempotence et protection contre les doublons.

Redis ne remplace pas PostgreSQL. Il complète l'architecture.

## 2. Modèle mental de Redis

Redis est un magasin de données en mémoire, principalement utilisé via des clés. Une clé peut pointer vers plusieurs types de structures :

| Type Redis | Usage AI Engineering |
|---|---|
| String | réponse LLM sérialisée, compteur, flag |
| Hash | métadonnées de session ou profil court |
| List | historique court de messages |
| Set | identifiants uniques, déduplication |
| Sorted Set | fenêtre glissante de rate limiting |
| Stream | file d'événements ou pipeline léger |

Le concept le plus important pour cette journée est le TTL.

Un TTL, ou time to live, définit combien de temps une donnée reste valide. Dans une plateforme IA, le TTL est une décision métier autant qu'une décision technique.

Exemples :

| Donnée | TTL typique | Justification |
|---|---:|---|
| Réponse de classification stable | 24 h à 7 jours | résultat souvent réutilisable |
| Réponse de génération créative | 5 min à 1 h | contexte et attentes variables |
| Session conversationnelle courte | 30 min à 2 h | état temporaire |
| Compteur de rate limit | 1 min à 24 h | contrôle opérationnel |
| Idempotency key | 5 min à 24 h | éviter les doublons |

## 3. Redis dans l'architecture de la semaine 5

Le jour 2 a introduit l'API FastAPI. Le jour 3 ajoute Redis entre l'API et les services coûteux ou fragiles.

```mermaid
sequenceDiagram
    participant U as Utilisateur
    participant A as FastAPI
    participant R as Redis
    participant M as Modèle LLM

    U->>A: POST /chat
    A->>R: Vérifie rate limit
    R-->>A: Autorisé
    A->>R: Cherche réponse en cache
    alt Cache hit
        R-->>A: Réponse cachée
        A-->>U: Réponse rapide
    else Cache miss
        A->>M: Appel LLM
        M-->>A: Réponse générée
        A->>R: Stocke avec TTL
        A-->>U: Réponse
    end
```

La logique importante est la suivante : Redis protège le chemin critique, mais l'API doit continuer à exprimer les règles métier. Redis ne doit pas devenir un endroit où la logique produit est cachée.

## 4. Stratégie de nommage des clés

Un système Redis sans convention devient vite illisible. Chaque clé doit exprimer :

## Démonstration — clé de cache stable

In [ ]:
import hashlib
import json
from typing import Any

def canonical_json(payload: dict[str, Any]) -> str:
    return json.dumps(payload, sort_keys=True, ensure_ascii=False, separators=(",", ":"))

def build_llm_cache_key(
    app: str,
    env: str,
    model: str,
    prompt_version: str,
    payload: dict[str, Any],
) -> str:
    digest = hashlib.sha256(canonical_json(payload).encode("utf-8")).hexdigest()[:16]
    return f"{app}:{env}:cache:llm:{model}:{prompt_version}:hash:{digest}"

payload_a = {"text": "Bonjour", "task": "classify"}
payload_b = {"task": "classify", "text": "Bonjour"}

key_a = build_llm_cache_key("ai-api", "dev", "demo-llm", "v1", payload_a)
key_b = build_llm_cache_key("ai-api", "dev", "demo-llm", "v1", payload_b)

print(key_a)
print(key_b)
assert key_a == key_b

## Démonstration — rate limiter sans Redis réel

Cette cellule simule le comportement `INCR` + `EXPIRE` pour comprendre la logique avant d'utiliser Redis dans le lab.

In [ ]:
class FakeRedis:
    def __init__(self):
        self.values = {}
        self.ttls = {}

    async def incr(self, key: str) -> int:
        self.values[key] = self.values.get(key, 0) + 1
        return self.values[key]

    async def expire(self, key: str, ttl_seconds: int) -> None:
        self.ttls[key] = ttl_seconds

async def allow_request(redis, key: str, limit: int, ttl_seconds: int) -> bool:
    count = await redis.incr(key)
    if count == 1:
        await redis.expire(key, ttl_seconds)
    return count <= limit

redis = FakeRedis()
key = "ai-api:dev:ratelimit:user:u_123:minute"

results = []
for _ in range(5):
    results.append(await allow_request(redis, key, limit=3, ttl_seconds=60))

results

# Exercices

## Exercice 1 — Concevoir des clés Redis

Pour chacun des cas suivants, proposez une clé Redis conforme à la convention :

```text
{app}:{env}:{domain}:{entity}:{id}
```

Cas :

1. cache d'une réponse LLM pour une classification de ticket ;
2. session conversationnelle d'un utilisateur ;
3. compteur de rate limit par minute ;
4. clé d'idempotence pour une requête HTTP ;
5. verrou court pour empêcher deux workers de traiter le même job.

Livrable attendu : cinq clés et une justification courte.

## Exercice 2 — Construire une clé de cache stable

Écrivez une fonction Python :

```python
def build_cache_key(app: str, env: str, model: str, prompt_version: str, payload: dict) -> str:
    ...
```

Contraintes :

- la clé doit être stable si les champs du dictionnaire sont dans un ordre différent ;
- la clé ne doit pas contenir le prompt complet ;
- la clé doit inclure le modèle et la version du prompt ;
- le hash doit utiliser SHA-256.

Livrable attendu : fonction Python et deux exemples montrant que l'ordre des clés ne change pas le résultat.

## Exercice 3 — Choisir des TTL

Proposez un TTL pour chaque donnée :

| Donnée | TTL à proposer |
|---|---|
| résumé d'un document interne non sensible |
| réponse de chat très contextuelle |
| session conversationnelle courte |
| compteur de rate limit minute |
| clé d'idempotence pour création de job |
| résultat de classification de ticket |

Livrable attendu : tableau complété et justification.

## Exercice 4 — Implémenter un rate limiter simple

Écrivez une fonction asynchrone :

```python
async def allow_request(redis, key: str, limit: int, ttl_seconds: int) -> bool:
    ...
```

Contraintes :

- utiliser `INCR` ;
- poser le TTL uniquement au premier incrément ;
- retourner `True` si la requête est autorisée ;
- retourner `False` si la limite est dépassée.

Livrable attendu : fonction Python + explication du comportement.


# Challenge — Couche Redis pour API IA

## Contexte

Vous travaillez sur une API FastAPI qui expose un endpoint `/chat`. Chaque requête contient :

- un `user_id` ;
- un `conversation_id` ;
- un message utilisateur ;
- un nom de modèle ;
- une version de prompt système.

L'appel au modèle est coûteux. L'API doit être protégée contre les abus et doit conserver un historique conversationnel court.

## Objectif

Concevoir et implémenter une couche Redis qui fournit :

1. un cache de réponse LLM ;
2. une session conversationnelle courte ;
3. un rate limiter par utilisateur ;
4. une stratégie de clés claire ;
5. une politique de TTL ;
6. un comportement dégradé si Redis est indisponible.

## Contraintes

- Le cache doit utiliser SHA-256.
- La session doit expirer.
- Le rate limiter doit utiliser `INCR` + `EXPIRE`.
- Les clés doivent être préfixées par application et environnement.
- Les données sensibles ne doivent pas apparaître en clair dans les clés.
- La solution doit être compatible avec FastAPI.
- Le code doit être compréhensible par un AI Backend Engineer junior.

## Livrables attendus

1. diagramme Mermaid de l'architecture ;
2. table des clés Redis ;
3. code Python de la couche cache ;
4. code Python du rate limiter ;
5. exemple d'intégration dans un endpoint FastAPI ;
6. explication des choix de TTL ;
7. limites de la solution.

## Barème indicatif

| Critère | Points |
|---|---:|
| Stratégie de clés | 20 |
| Cache LLM stable | 20 |
| Session courte avec TTL | 15 |
| Rate limiter correct | 15 |
| Intégration FastAPI | 15 |
| Résilience et sécurité | 10 |
| Clarté de l'explication | 5 |


# Références

## Documentation technique à consulter

- Redis documentation — commandes `GET`, `SET`, `INCR`, `EXPIRE`, structures de données et TTL.
- redis-py documentation — client Python asynchrone `redis.asyncio`.
- FastAPI documentation — dépendances, endpoints asynchrones et gestion des erreurs HTTP.
- Docker Compose documentation — orchestration locale d'un service Redis et d'une API.
- OWASP Cheat Sheet Series — stockage de données sensibles, logs et secrets.

## Concepts à maîtriser

- Cache-aside pattern.
- TTL et invalidation de cache.
- Rate limiting par fenêtre fixe.
- Idempotency key.
- Session state court terme.
- Différence entre stockage volatil et stockage durable.
- Dégradation contrôlée d'un service de production.

## Pour aller plus loin

- Fenêtre glissante avec sorted sets Redis.
- Redis Streams pour pipelines d'événements.
- Redis Sentinel et Redis Cluster.
- Circuit breaker autour des services d'infrastructure.
- Mesure du cache hit rate par endpoint.
